# 融合错误分析 — 各位置判错了哪些样本

在真实 XD-Violence **3 模态**数据（视觉 I3D + 音频 AST + 文本 ASR）上，用**电影分组
交叉验证**为每个片段拿到 **out-of-fold（OOF）预测**，然后把**被判错的样本**挑出来看：

- 每个融合位置（单模态 / ①②③④⑤⑥）各自错在哪、错多少
- **所有方法都判错**的"最难"样本（真·模糊）
- **① early vs ⑤ late 的分歧样本**（深融合帮到 / 帮倒忙的地方）
- **跨模态冲突**样本（模态互相矛盾）——各融合怎么裁决

> 片段名本身就是线索：`电影名.年份__#起止时间_label_X`（`label_A`=正常，其余=暴力）。
> 前置：在 GPU 机器上跑，`data/xd-violence/` 下要有 i3d_rgb / audio_full / text_features。

In [1]:
import glob, os, logging, warnings
from collections import defaultdict
import numpy as np, pandas as pd, torch
# 让 Lightning 别把 GPU/TPU 横幅刷进每个 cell 的输出
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
from torch import nn

from sentinelai.coordinated_fusion import CoordinatedFusion
from sentinelai.early_fusion import JointFusionTransformer
from sentinelai.train.lit_module import LitCrossAttention
import lightning.pytorch as pl
from torch.utils.data import DataLoader, TensorDataset

DATA = os.path.expanduser("~/documents/SentinelAI/data/xd-violence")
I3D = f"{DATA}/data/i3d_rgb"
DIRS = ["1-1004", "1005-2004", "2005-2804", "2805-3319", "3320-3954", "test_videos"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
V_TOK, A_TOK = 32, 16
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 46)

def b(n): return n.replace(".npy", "").replace(".mp4", "")
def is_violent(n): return 0 if "_label_A" in n else 1
def resample(x, n):
    if len(x) == 0: return np.zeros((n, x.shape[1]), np.float32)
    return x[np.linspace(0, len(x) - 1, n).round().astype(int)].astype(np.float32)

## 1. 载入三模态特征（保留每片段的 key，用于回看错例）

In [2]:
def load_i3d():
    out = {}
    for d in DIRS:
        for f in glob.glob(f"{I3D}/{d}/*.npy"):
            a = np.load(f); a = a.mean(1) if a.ndim == 3 else a
            out[b(os.path.basename(f))] = a.astype(np.float32)
    return out
def load_npz(folder):
    out = {}
    for f in glob.glob(f"{folder}/*.npz"):
        z = np.load(f, allow_pickle=True); out[str(z["key"])] = z["embedding"].astype(np.float32)
    return out

vis, aud, txt = load_i3d(), load_npz(f"{DATA}/audio_full"), load_npz(f"{DATA}/text_features")
keys = sorted(k for k in vis if k in aud and k in txt)
y = np.array([is_violent(k) for k in keys])
groups = np.array([k.split("__")[0] for k in keys])
movie = [k.split("__")[0] for k in keys]
print(f"{len(keys)} 个 3 模态片段, {len(set(groups))} 部电影, {y.mean():.0%} 暴力")

Vp = np.stack([vis[k].mean(0) for k in keys])
Ap = np.stack([aud[k] for k in keys])
Tp = np.stack([txt[k] for k in keys])
Vs = np.stack([resample(vis[k], V_TOK) for k in keys])
allp = np.concatenate([Vp, Ap, Tp], 1)
cv = list(GroupKFold(5).split(np.arange(len(keys)), y, groups))

774 个 3 模态片段, 215 部电影, 49% 暴力


## 2. 每个位置的 out-of-fold 预测
单模态 + ③拼接 + ④决策树 + ⑤平均 用 sklearn 的 `cross_val_predict`（同一 CV）；
①early / ②coordinated / ⑥cross-attn 是 torch/lightning，手动在每折训练、对测试折出概率。

In [3]:
def sk_oof(X):
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    return cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]

pv, pa, pt = sk_oof(Vp), sk_oof(Ap), sk_oof(Tp)          # 单模态
p3 = sk_oof(allp)                                        # ③ 拼接
p5 = (pv + pa + pt) / 3                                  # ⑤ 平均

# ④ 决策级：GBDT stack 各模态概率（用内层 OOF 概率训练，防泄漏）
p4 = np.zeros(len(keys))
for tr, te in cv:
    g = groups[tr]
    def inner(X):
        pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
        return cross_val_predict(pipe, X[tr], y[tr], cv=list(GroupKFold(3).split(X[tr], y[tr], g)), method="predict_proba")[:, 1]
    meta = GradientBoostingClassifier(random_state=0).fit(np.c_[inner(Vp), inner(Ap), inner(Tp)], y[tr])
    p4[te] = meta.predict_proba(np.c_[pv[te], pa[te], pt[te]])[:, 1]

In [4]:
def torch_oof_early_coord(kind):
    """① early 或 ② coordinated 的 OOF 概率。"""
    Ts = Tp[:, None, :]; As = Ap[:, None, :]           # 文本/音频各作 1 token
    out = np.zeros(len(keys)); dims = {"visual": 2048, "audio": 768, "text": 768}
    for tr, te in cv:
        torch.manual_seed(0)
        model = (JointFusionTransformer(dims, d_model=128, n_layers=2, n_categories=1) if kind == "early"
                 else CoordinatedFusion(dims, d_model=128, n_categories=1)).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), 1e-3); lf = nn.BCEWithLogitsLoss()
        if kind == "early":
            tr_in = {"visual": Vs[tr], "audio": As[tr], "text": Ts[tr]}; te_in = {"visual": Vs[te], "audio": As[te], "text": Ts[te]}
        else:
            tr_in = {"visual": Vp[tr], "audio": Ap[tr], "text": Tp[tr]}; te_in = {"visual": Vp[te], "audio": Ap[te], "text": Tp[te]}
        yt = torch.tensor(y[tr], dtype=torch.float32, device=DEVICE)[:, None]
        T = {m: torch.tensor(v, device=DEVICE) for m, v in tr_in.items()}
        E = {m: torch.tensor(v, device=DEVICE) for m, v in te_in.items()}
        for _ in range(200):
            model.train(); opt.zero_grad(); o = model(T); lo = o[0] if isinstance(o, tuple) else o
            lf(lo, yt).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            o = model(E); lo = o[0] if isinstance(o, tuple) else o
            out[te] = torch.sigmoid(lo).squeeze(1).cpu().numpy()
    return out

p1 = torch_oof_early_coord("early")
p2 = torch_oof_early_coord("coord")

In [5]:
def crossattn_oof():
    """⑥ cross-attn：I3D 帧作 K/V，[音频,文本] 作 Query。"""
    G = np.stack([Ap, Tp], 1)                          # (N,2,768) 两个 query token
    out = np.zeros(len(keys))
    for tr, te in cv:
        pl.seed_everything(0, verbose=False)
        lit = LitCrossAttention(video_dim=2048, guide_dim=768, n_categories=1, d_model=128, n_heads=4, lr=1e-3)
        ds = TensorDataset(torch.tensor(Vs[tr]), torch.tensor(G[tr]), torch.tensor(y[tr][:, None], dtype=torch.float32))
        pl.Trainer(max_epochs=50, accelerator="auto", devices=1, logger=False, enable_checkpointing=False,
                   enable_progress_bar=False, enable_model_summary=False, limit_val_batches=0,
                   num_sanity_val_steps=0).fit(lit, DataLoader(ds, batch_size=128, shuffle=True))
        lit.eval()
        with torch.no_grad():
            out[te] = lit.model.predict_proba(torch.tensor(Vs[te], device=lit.device),
                                              torch.tensor(G[te], device=lit.device))[:, 0].cpu().numpy()
    return out

p6 = crossattn_oof()

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


HPU available: False, using: 0 HPUs


You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


`Trainer.fit` stopped: `max_epochs=50` reached.


## 3. 汇总成一张表，算各方法的错误数

In [6]:
probs = {"visual": pv, "audio": pa, "text": pt, "①early": p1, "②coord": p2,
         "③concat": p3, "④gbdt": p4, "⑤late": p5, "⑥xattn": p6}
df = pd.DataFrame({"key": keys, "movie": movie, "label": y})
for m, p in probs.items():
    df[m] = p; df[m + "_pred"] = (p >= 0.5).astype(int)

rows = []
for m, p in probs.items():
    pred = (p >= 0.5).astype(int)
    rows.append({"方法": m, "F1": f1_score(y, pred), "P": precision_score(y, pred),
                 "R": recall_score(y, pred), "错误数": int((pred != y).sum()),
                 "漏报(FN)": int(((pred == 0) & (y == 1)).sum()),
                 "误报(FP)": int(((pred == 1) & (y == 0)).sum())})
summary = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
summary.round(3)

,方法,F1,P,R,错误数,漏报(FN),误报(FP)
0,④gbdt,0.943,0.946,0.939,43,23,20
1,②coord,0.941,0.923,0.960,45,15,30
2,⑥xattn,0.941,0.930,0.952,45,18,27
3,③concat,0.939,0.941,0.936,46,24,22
4,①early,0.937,0.923,0.952,48,18,30
5,⑤late,0.936,0.939,0.934,48,25,23
6,audio,0.928,0.928,0.928,54,27,27
7,visual,0.905,0.907,0.904,71,36,35
8,text,0.745,0.768,0.723,186,104,82


## 3.5 误分类明细表（长表：一条错例一行）
每个**被判错的样本**一行，带上：**方法（融合种类）**、**错误类型**（FN 漏报 / FP 误报）、
样本本身（key、电影）、以及各模态/该方法的概率。可按 `方法` 或 `错误类型` 任意筛选来分析。

In [7]:
records = []
for m, p in probs.items():
    pred = (p >= 0.5).astype(int)
    for i in np.where(pred != y)[0]:
        records.append({
            "方法": m,
            "错误类型": "FN漏报" if y[i] == 1 else "FP误报",
            "key": keys[i], "电影": movie[i],
            "真实": int(y[i]), "预测": int(pred[i]), "该方法概率": round(float(p[i]), 2),
            "视觉": round(float(pv[i]), 2), "音频": round(float(pa[i]), 2), "文本": round(float(pt[i]), 2),
        })
mis = pd.DataFrame(records)
print(f"总误分类记录：{len(mis)} 条（跨 {len(probs)} 种方法、{mis['key'].nunique()} 个不同片段）")
# 每种方法的 FN / FP 分布
mis.groupby(["方法", "错误类型"]).size().unstack(fill_value=0)

总误分类记录：586 条（跨 9 种方法、265 个不同片段）


错误类型,FN漏报,FP误报
方法,,
audio,27,27
text,104,82
visual,36,35
①early,18,30
②coord,15,30
③concat,24,22
④gbdt,23,20
⑤late,25,23
⑥xattn,18,27


明细样本（默认按 电影+片段 排序，方便看同一片段被哪些方法、以什么类型判错）。
换筛选看具体情况，例如：
`mis[mis["方法"] == "⑤late"]`　或　`mis[mis["错误类型"] == "FP误报"]`　或　`mis[mis["电影"].str.contains("...")]`

In [8]:
mis.sort_values(["电影", "key", "方法"]).head(40)

,方法,错误类型,key,电影,真实,预测,该方法概率,视觉,音频,文本
125,text,FP误报,A.Beautiful.Mind.2001__#01-26-10_01-28-00_...,A.Beautiful.Mind.2001,0,1,0.98,0.00,0.00,0.98
126,text,FP误报,A.Beautiful.Mind.2001__#01-29-31_01-31-00_...,A.Beautiful.Mind.2001,0,1,0.51,0.00,0.00,0.51
127,text,FP误报,About.Time.2013__#00-17-30_00-19-10_label_A,About.Time.2013,0,1,0.59,0.00,0.00,0.59
0,visual,FP误报,About.Time.2013__#00-20-35_00-23-06_label_A,About.Time.2013,0,1,0.72,0.72,0.01,0.07
541,⑥xattn,FN漏报,Bad.Boys.II.2003__#00-43-05_00-44-20_label...,Bad.Boys.II.2003,1,0,0.21,0.86,0.98,0.75
128,text,FP误报,Bad.Boys.II.2003__#01-46-00_01-48-35_label_A,Bad.Boys.II.2003,0,1,0.56,0.00,0.03,0.56
129,text,FP误报,Bad.Boys.II.2003__#02-05-15_02-05-36_label_A,Bad.Boys.II.2003,0,1,0.68,1.00,0.05,0.68
1,visual,FP误报,Bad.Boys.II.2003__#02-05-15_02-05-36_label_A,Bad.Boys.II.2003,0,1,1.00,1.00,0.05,0.68
311,①early,FP误报,Bad.Boys.II.2003__#02-05-15_02-05-36_label_A,Bad.Boys.II.2003,0,1,1.00,1.00,0.05,0.68
493,⑤late,FP误报,Bad.Boys.II.2003__#02-05-15_02-05-36_label_A,Bad.Boys.II.2003,0,1,0.58,1.00,0.05,0.68


## 4. 所有方法都判错的"最难"样本
每个片段被多少种方法判错；被 9 种全判错的，是真·模糊/标注可疑的样本。

In [9]:
pred_cols = [m + "_pred" for m in probs]
df["错误方法数"] = sum((df[c] != df["label"]) for c in pred_cols)
hard = df[df["错误方法数"] == len(probs)][["key", "label", "错误方法数"] + list(probs)]
print(f"被全部 {len(probs)} 种方法判错的样本：{len(hard)} 个")
hard.round(2)

被全部 9 种方法判错的样本：6 个


,key,label,错误方法数,visual,audio,text,①early,②coord,③concat,④gbdt,⑤late,⑥xattn
87,Braveheart.1995__#02-43-01_02-46-50_label_...,1,9,0.02,0.01,0.48,0.00,0.00,0.00,0.03,0.17,0.00
101,Bullet.in.the.Head.1990__#00-04-41_00-06-1...,1,9,0.00,0.00,0.36,0.00,0.00,0.00,0.00,0.12,0.00
115,Bullet.in.the.Head.1990__#01-44-20_01-45-2...,0,9,1.00,0.51,0.59,1.00,1.00,1.00,0.79,0.70,1.00
158,Deadpool.2016__#0-17-08_0-17-52_label_A,0,9,0.99,0.95,0.80,1.00,1.00,1.00,0.85,0.91,1.00
178,Election.2005__#01-08-01_01-09-04_label_B1...,1,9,0.03,0.01,0.40,0.00,0.01,0.00,0.01,0.15,0.01
571,The.Notebook.2004__#00-23-50_00-24-31_label_A,0,9,0.98,0.71,0.78,0.99,1.00,0.99,0.90,0.82,1.00


## 5. ① early vs ⑤ late 的分歧样本
交叉点实验里 ① early 数据够时反超 ⑤ late。看具体是哪些片段让它们分道扬镳。

In [10]:
ea = (df["①early_pred"] == df["label"]) & (df["⑤late_pred"] != df["label"])
le = (df["⑤late_pred"] == df["label"]) & (df["①early_pred"] != df["label"])
print(f"① 对而 ⑤ 错：{ea.sum()} 个　|　⑤ 对而 ① 错：{le.sum()} 个")
print("\n— ① early 救回来的（⑤ 错、① 对）—")
display_cols = ["key", "label", "visual", "audio", "text", "①early", "⑤late"]
df[ea][display_cols].round(2).head(12)

① 对而 ⑤ 错：24 个　|　⑤ 对而 ① 错：24 个

— ① early 救回来的（⑤ 错、① 对）—


,key,label,visual,audio,text,①early,⑤late
78,Braveheart.1995__#00-58-01_01-00-01_label_...,1,0.00,0.04,0.12,1.00,0.06
85,Braveheart.1995__#02-19-30_02-20-55_label_...,1,0.02,0.95,0.36,1.00,0.44
94,Brick.Mansions.2014__#00-38-48_00-39-22_la...,0,0.88,0.05,0.80,0.00,0.58
128,City.Of.Men.2007__#01-10-21_01-11-35_label_A,0,1.00,0.00,0.57,0.08,0.52
138,City.of.God.2002__#01-35-27_01-36-12_label...,1,1.00,0.08,0.40,0.93,0.49
160,Deadpool.2016__#0-28-34_0-29-54_label_B1-0-0,1,1.00,0.29,0.05,1.00,0.45
169,Death.Proof.2007__#01-38-30_01-40-16_label_A,0,0.00,0.89,0.98,0.00,0.62
171,Desperado.1995__#00-04-53_00-07-53_label_B...,1,0.07,0.96,0.26,1.00,0.43
181,Fast.Five.2011__#00-00-50_00-01-20_label_A,0,0.51,1.00,0.97,0.01,0.83
208,Fast.Furious.6.2013__#00-44-20_00-45-20_la...,1,0.08,1.00,0.11,1.00,0.40


In [11]:
print("— ⑤ late 更稳的（① 错、⑤ 对）—")
df[le][display_cols].round(2).head(12)

— ⑤ late 更稳的（① 错、⑤ 对）—


,key,label,visual,audio,text,①early,⑤late
69,Black.Hawk.Down.2001__#00-12-00_00-12-31_l...,0,0.00,0.14,1.00,1.00,0.38
141,Crank.Dircut.2006__#0-41-11_0-42-10_label_...,1,1.00,0.00,0.77,0.00,0.59
185,Fast.Five.2011__#00-49-45_00-49-58_label_B...,1,0.98,0.45,0.85,0.01,0.76
191,Fast.Five.2011__#01-47-02_01-47-34_label_B...,1,0.59,1.00,0.16,0.03,0.58
207,Fast.Furious.6.2013__#00-41-00_00-41-40_la...,1,0.96,0.95,0.73,0.16,0.88
230,Fury.2014__#00-01-50_00-02-58_label_A,0,0.00,0.01,0.92,1.00,0.31
241,Gladiator.2000__#00-13-32_00-14-40_label_A,0,0.75,0.08,0.38,1.00,0.40
244,Gladiator.2000__#01-27-58_01-29-40_label_A,0,0.83,0.03,0.56,1.00,0.47
265,GoldenEye.1995__#02-05-18_02-06-13_label_A,0,0.79,0.00,0.44,1.00,0.41
293,Hot.Fuzz.2007__#00-14-20_00-15-05_label_A,0,1.00,0.11,0.21,1.00,0.44


## 6. 跨模态冲突样本 —— 模态互相矛盾时各融合怎么裁决
挑出**单模态判断互相打架**的片段（有的模态说暴力、有的说正常），看各融合的最终裁决。
文本最弱（对物理暴力几乎无信号），这里最容易看到"融合要不要信文本"。

In [12]:
single = df[["visual", "audio", "text"]].values
disagree = (single >= 0.5).sum(1)                      # 0..3 个模态投暴力
df["模态分歧"] = np.minimum(disagree, 3 - disagree)   # 1 = 最分歧(2:1)，0 = 一致
conflict = df[df["模态分歧"] == 1].copy()
print(f"三模态出现 2:1 分歧的片段：{len(conflict)} 个")
cols = ["key", "label", "visual", "audio", "text", "②coord", "④gbdt", "⑤late", "⑥xattn"]
conflict.sort_values("text")[cols].round(2).head(15)

三模态出现 2:1 分歧的片段：240 个


,key,label,visual,audio,text,②coord,④gbdt,⑤late,⑥xattn
341,Jason.Bourne.2016__#01-16-40_01-18-50_label_A,0,0.00,0.54,0.02,0.00,0.01,0.19,0.00
769,v=idBwMN0E9Qg__#00-18-44_00-24-16_label_B4...,1,0.92,0.07,0.02,0.71,0.18,0.34,1.00
737,v=KeJw6U9_VeY__#1_label_B4-0-0,1,1.00,1.00,0.03,1.00,1.00,0.67,1.00
231,Fury.2014__#00-09-04_00-10-42_label_A,0,0.99,0.23,0.03,0.94,0.43,0.42,1.00
711,v=CoxsDkB-rWU__#00-01-20_00-07-06_label_B4...,1,1.00,0.74,0.03,1.00,0.92,0.59,1.00
427,Mission.Impossible.II.2000__#01-22-53_01-2...,1,1.00,1.00,0.03,1.00,0.99,0.68,1.00
269,Good.Will.Hunting.1997__#00-16-18_00-18-09...,0,0.95,0.00,0.04,0.00,0.40,0.33,0.00
751,v=dOUiP6CClqc__#1_label_B4-0-0,1,1.00,1.00,0.04,1.00,0.98,0.68,1.00
255,GoldenEye.1995__#00-26-45_00-27-14_label_B...,1,0.31,0.97,0.04,0.97,0.58,0.44,0.00
160,Deadpool.2016__#0-28-34_0-29-54_label_B1-0-0,1,1.00,0.29,0.05,0.88,0.79,0.45,1.00


## 7. 文本把 ⑤ 平均带偏的例子
文本是最弱模态。挑"视觉+音频都对、但文本强烈反向"的片段，看 naive 平均 ⑤ 是否被文本拖错、
而学习式融合（②/④/⑥）有没有顶住。

In [13]:
va_right = ((df["visual"] >= 0.5) == df["label"]) & ((df["audio"] >= 0.5) == df["label"])
text_wrong = (df["text"] >= 0.5) != df["label"]
misled = df[va_right & text_wrong]
print(f"视觉+音频对、文本反向的片段：{len(misled)} 个")
print(f"其中 ⑤late 被带错：{int(((misled['⑤late']>=0.5)!=misled['label']).sum())} 个"
      f"　②coord 被带错：{int(((misled['②coord']>=0.5)!=misled['label']).sum())} 个"
      f"　④gbdt 被带错：{int(((misled['④gbdt']>=0.5)!=misled['label']).sum())} 个")
misled[["key", "label", "visual", "audio", "text", "②coord", "④gbdt", "⑤late", "⑥xattn"]].round(2).head(12)

视觉+音频对、文本反向的片段：140 个
其中 ⑤late 被带错：0 个　②coord 被带错：2 个　④gbdt 被带错：2 个


,key,label,visual,audio,text,②coord,④gbdt,⑤late,⑥xattn
9,A.Beautiful.Mind.2001__#01-26-10_01-28-00_...,0,0.00,0.00,0.98,0.00,0.01,0.33,0.00
10,A.Beautiful.Mind.2001__#01-29-31_01-31-00_...,0,0.00,0.00,0.51,0.00,0.01,0.17,0.00
14,About.Time.2013__#00-17-30_00-19-10_label_A,0,0.00,0.00,0.59,0.00,0.03,0.20,0.00
35,Bad.Boys.II.2003__#01-46-00_01-48-35_label_A,0,0.00,0.03,0.56,0.00,0.05,0.20,0.00
39,Be.with.You.2018__#00-08-55_00-10-50_label_A,0,0.00,0.00,0.60,0.00,0.00,0.20,0.00
69,Black.Hawk.Down.2001__#00-12-00_00-12-31_l...,0,0.00,0.14,1.00,0.06,0.10,0.38,0.54
72,Black.Hawk.Down.2001__#01-23-28_01-24-47_l...,1,0.52,0.68,0.39,0.99,0.45,0.53,1.00
73,Black.Hawk.Down.2001__#01-49-18_01-49-50_l...,1,0.87,1.00,0.28,1.00,0.99,0.72,1.00
76,Black.Hawk.Down.2001__#02-14-30_02-14-50_l...,0,0.00,0.00,0.93,0.00,0.00,0.31,0.00
81,Braveheart.1995__#01-38-50_01-39-49_label_...,1,1.00,1.00,0.44,1.00,1.00,0.81,1.00


## 结论（看数字填）
- 各方法错误数见 §3；全判错的"硬样本"见 §4（多半是镜头模糊/标注边界）。
- §5 显示 ① early 和 ⑤ late 各自救回的片段，解释交叉点为什么发生在样本层面。
- §6/§7 显示**弱文本冲突**时，naive 平均 ⑤ 容易被带偏，而能学权重的 ②/④/⑥ 更稳——
  这正是"学习式融合对模态质量差异鲁棒"在**单个错例**上的体现。